# ⭐ Day 89: Generative AI - Building Your First GAN (Generative Adversarial Networks)
### Day 89 of 369-day Python & AI Learning Path

🎨 Welcome to Day 89 of your incredible 369-day journey! Today, we step into one of the most exciting and transformative domains in all of Artificial Intelligence: **Generative AI**. We will build a **Generative Adversarial Network (GAN)** from scratch, training two neural networks—the Generator and the Discriminator—to compete against each other in a beautiful game of creation and critique. By the end of this notebook, you will generate realistic images and understand the foundational architecture behind modern AI image generation. Let's create something amazing! 🚀

## 📚 Table of Contents
1. [Introduction to Generative AI and GANs](#1)
2. [Understanding the GAN Framework](#2)
3. [Loading and Preprocessing Image Dataset](#3)
4. [Building the Generator Network](#4)
5. [Building the Discriminator Network](#5)
6. [Training the GAN](#6)
7. [Visualizing Generated Images](#7)
8. [Evaluating GAN Performance](#8)
9. [Tips for Stable GAN Training](#9)
10. [Modern Variants](#10)
11. [Hands-On Exercises](#11)
12. [Solutions](#12)
13. [Summary & Day 90 Teaser](#13)

<a id='1'></a>
## 🎨 1. Introduction to Generative AI and GANs

**Generative AI** refers to models that can create new data—images, text, music, and more—that resembles real data. Unlike discriminative models that classify inputs, generative models learn the underlying distribution of the data and sample from it.

### What is a GAN?
A **Generative Adversarial Network (GAN)**, introduced by Ian Goodfellow in 2014, consists of two neural networks:
- **Generator (G)**: Creates fake data from random noise.
- **Discriminator (D)**: Distinguishes between real and fake data.

They are trained simultaneously in a **minimax game**: the Generator tries to fool the Discriminator, while the Discriminator tries to catch the Generator. This adversarial process drives both networks to improve continuously.

💡 **Why GANs matter?** GANs power deepfakes, style transfer, super-resolution, drug discovery, and even art generation. They are the ancestors of modern diffusion models!

<a id='2'></a>
## 🧠 2. Understanding the GAN Framework

### The Minimax Game
The objective function for a GAN is:

$$ \min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z)))] $$

Where:
- $x$ = real data sample
- $z$ = random noise vector (latent space)
- $D(x)$ = probability that $x$ is real
- $G(z)$ = generated fake sample

### Training Dynamics
| Phase | What Happens |
|-------|-------------|
| **Discriminator Training** | Trained on real images (label=1) and fake images (label=0) |
| **Generator Training** | Trained to make the Discriminator classify fakes as real (label=1) |

🚀 This adversarial tension is what makes GANs so powerful—and sometimes so tricky to train!

<a id='3'></a>
## 📦 3. Loading and Preprocessing Image Dataset

We will use **Fashion-MNIST**—a more interesting dataset than standard MNIST, containing 60,000 grayscale images of 10 fashion categories (28×28 pixels).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from IPython.display import clear_output

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"🚀 GPU Available: {tf.config.list_physical_devices('GPU')}")

# Load Fashion-MNIST dataset
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize images to [-1, 1] range (important for GANs!)
X_train = (X_train.astype('float32') - 127.5) / 127.5
X_train = np.expand_dims(X_train, axis=-1)  # Add channel dimension: (60000, 28, 28, 1)

print(f"📊 Training data shape: {X_train.shape}")
print(f"📊 Data range: [{X_train.min():.2f}, {X_train.max():.2f}]")

# Class names for reference
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Visualize sample real images
plt.figure(figsize=(12, 4))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(X_train[i].squeeze(), cmap='gray')
    plt.title(class_names[y_train[i]], fontsize=8)
    plt.axis('off')
plt.suptitle('👕 Real Fashion-MNIST Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='4'></a>
## 🏗️ 4. Building the Generator Network

The Generator transforms random noise vectors into realistic 28×28 images. We use **Transposed Convolutions** (also called Deconvolutions) to upsample from a small latent vector to a full image.

**Architecture:**
- Input: 100-dimensional noise vector
- Dense → Reshape → Conv2DTranspose layers
- Output: 28×28×1 image with tanh activation

In [ ]:
def build_generator(latent_dim=100):
    """
    Builds the Generator network.
    Transforms a latent vector into a 28x28x1 image.
    """
    model = keras.Sequential(name="Generator")
    
    # Foundation: 7x7x256 feature maps
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Reshape((7, 7, 256)))
    assert model.output_shape == (None, 7, 7, 256)
    
    # Upsample to 14x14x128
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    
    # Upsample to 14x14x64
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    
    # Output: 28x28x1 with tanh activation
    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 28, 28, 1)
    
    return model

# Instantiate and summarize the Generator
latent_dim = 100
generator = build_generator(latent_dim)
generator.summary()

# Test: Generate a sample image from random noise
noise = tf.random.normal([1, latent_dim])
generated_image = generator(noise, training=False)

plt.figure(figsize=(4, 4))
plt.imshow(generated_image[0, :, :, 0], cmap='gray')
plt.title('🎲 Untrained Generator Output', fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()

<a id='5'></a>
## 🔍 5. Building the Discriminator Network

The Discriminator is a binary classifier that takes an image and outputs the probability that it is real (as opposed to generated).

**Architecture:**
- Input: 28×28×1 image
- Conv2D layers with LeakyReLU and Dropout for regularization
- Flatten → Dense with sigmoid activation
- Output: Single scalar probability [0, 1]

In [ ]:
def build_discriminator():
    """
    Builds the Discriminator network.
    Classifies images as real (1) or fake (0).
    """
    model = keras.Sequential(name="Discriminator")
    
    # Input: 28x28x1
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))
    
    # Downsample
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))
    
    # Flatten and classify
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    
    return model

# Instantiate and summarize the Discriminator
discriminator = build_discriminator()
discriminator.summary()

# Test: Classify the untrained generated image
decision = discriminator(generated_image)
print(f"🔍 Discriminator decision on untrained fake image: {decision.numpy()[0][0]:.4f}")
print(f"   (Closer to 0 = fake, closer to 1 = real)")

<a id='6'></a>
## ⚔️ 6. Training the GAN

This is the heart of the GAN! We define the loss functions, optimizers, and a custom training loop. The key challenge is balancing the training of both networks so neither dominates.

### Loss Functions
- **Discriminator Loss**: Binary cross-entropy on real vs fake classifications
- **Generator Loss**: Binary cross-entropy where the Generator wants the Discriminator to classify fakes as real (label=1)

In [ ]:
# Define loss and optimizers
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=False)

# Use different learning rates for stability
generator_optimizer = keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5)

def discriminator_loss(real_output, fake_output):
    """Binary cross-entropy loss for the Discriminator."""
    real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)  # Label smoothing
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    """Binary cross-entropy loss for the Generator."""
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# Create the full GAN model (for visualization, though we train separately)
# Training will be done step-by-step in the custom loop
print("✅ Loss functions and optimizers defined!")
print("💡 Using label smoothing (0.9 instead of 1.0) for more stable training.")

In [ ]:
# Training hyperparameters
EPOCHS = 50
BATCH_SIZE = 256
BUFFER_SIZE = 60000
noise_dim = 100
num_examples_to_generate = 16

# Prepare the dataset
train_dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

# Seed for consistent visualization during training
seed = tf.random.normal([num_examples_to_generate, noise_dim])

# Lists to track losses
generator_losses = []
discriminator_losses = []

@tf.function
def train_step(images):
    """Single training step for both Generator and Discriminator."""
    noise = tf.random.normal([BATCH_SIZE, noise_dim])
    
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # Generate fake images
        generated_images = generator(noise, training=True)
        
        # Discriminator outputs
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        
        # Calculate losses
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
    
    # Calculate gradients
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    
    # Apply gradients
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    
    return gen_loss, disc_loss

def generate_and_save_images(model, epoch, test_input):
    """Generate and display images during training."""
    predictions = model(test_input, training=False)
    
    fig = plt.figure(figsize=(8, 8))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.suptitle(f'🎨 Generated Images at Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("🚀 Starting GAN Training...")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Latent Dimension: {noise_dim}")
print("-" * 50)

# Training loop
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    epoch_gen_loss = []
    epoch_disc_loss = []
    
    for image_batch in train_dataset:
        gen_loss, disc_loss = train_step(image_batch)
        epoch_gen_loss.append(gen_loss)
        epoch_disc_loss.append(disc_loss)
    
    # Average losses for the epoch
    avg_gen_loss = np.mean(epoch_gen_loss)
    avg_disc_loss = np.mean(epoch_disc_loss)
    generator_losses.append(avg_gen_loss)
    discriminator_losses.append(avg_disc_loss)
    
    # Display progress every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        clear_output(wait=True)
        print(f"⏱️  Epoch {epoch}/{EPOCHS} completed in {time.time() - start_time:.2f}s")
        print(f"   Generator Loss: {avg_gen_loss:.4f} | Discriminator Loss: {avg_disc_loss:.4f}")
        generate_and_save_images(generator, epoch, seed)
    else:
        print(f"⏱️  Epoch {epoch}/{EPOCHS} | Gen Loss: {avg_gen_loss:.4f} | Disc Loss: {avg_disc_loss:.4f}")

print("\n🎉 Training Complete!")

<a id='7'></a>
## 📊 7. Visualizing Generated Images During Training

Let's create a comprehensive visualization dashboard showing:
1. Loss curves for both networks
2. A grid of final generated images
3. Comparison between real and generated images

In [ ]:
# Plot 1: Loss Curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(generator_losses, label='Generator Loss', color='#FF6B6B', linewidth=2)
plt.plot(discriminator_losses, label='Discriminator Loss', color='#4ECDC4', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('📈 GAN Training Loss Curves', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Plot 2: Final Generated Image Grid
plt.subplot(1, 2, 2)
final_noise = tf.random.normal([25, noise_dim])
final_images = generator(final_noise, training=False)

fig2, axes = plt.subplots(5, 5, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(final_images[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
    ax.axis('off')
fig2.suptitle('🎨 Final Generated Samples (5×5 Grid)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot 3: Real vs Generated Comparison
plt.figure(figsize=(14, 6))

# Real images
plt.subplot(1, 2, 1)
for i in range(16):
    plt.subplot(2, 8, i + 1)
    plt.imshow(X_train[i].squeeze() * 127.5 + 127.5, cmap='gray')
    plt.axis('off')
plt.suptitle('👕 Real Fashion-MNIST Images', fontsize=14, fontweight='bold', y=0.98)

# Generated images
plt.subplot(1, 2, 2)
for i in range(16):
    plt.subplot(2, 8, i + 1)
    plt.imshow(final_images[i].numpy().squeeze() * 127.5 + 127.5, cmap='gray')
    plt.axis('off')
plt.suptitle('🎨 Generated Images', fontsize=14, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

<a id='8'></a>
## 🎯 8. Evaluating GAN Performance (FID Score Concepts)

Evaluating GANs is notoriously difficult because there is no single ground truth. One of the most popular metrics is the **Fréchet Inception Distance (FID)**.

### What is FID?
FID measures the distance between the feature distributions of real and generated images using a pre-trained Inception network. Lower FID = better quality and diversity.

$$ FID = ||\mu_r - \mu_g||^2 + Tr(\Sigma_r + \Sigma_g - 2\sqrt{\Sigma_r \Sigma_g}) $$

Where $\mu$ and $\Sigma$ are the mean and covariance of the feature vectors.

💡 **Key Insight**: A good GAN should produce images that are both:
- **High Quality** (sharp, recognizable features)
- **High Diversity** (covers all modes of the data distribution)

### Practical Evaluation Tips
| Metric | What It Measures | Good Value |
|--------|-----------------|------------|
| FID Score | Distribution similarity | Lower is better (< 50 is decent) |
| Inception Score | Quality + Diversity | Higher is better (> 6 is good) |
| Visual Inspection | Human perception | Essential! |

✅ **Always visualize!** Quantitative metrics are helpful, but human judgment remains crucial for GAN evaluation.

<a id='9'></a>
## 💡 9. Tips for Stable GAN Training

GANs are famously difficult to train. Here are battle-tested strategies to achieve stability:

### 🔧 Architectural Tips
1. **Use LeakyReLU** instead of ReLU in both Generator and Discriminator
2. **Use Batch Normalization** in the Generator (but often NOT in the Discriminator's first layer)
3. **Avoid sparse gradients**: Use strided convolutions instead of pooling
4. **Use Transposed Convolutions carefully**—they can cause checkerboard artifacts

### ⚖️ Training Tips
1. **Label Smoothing**: Use 0.9 instead of 1.0 for real labels to prevent overconfidence
2. **Different Learning Rates**: Generator often needs a slightly higher LR
3. **Train Discriminator more**: Sometimes 3-5 Discriminator steps per Generator step
4. **Use Adam optimizer** with β₁=0.5 (not the default 0.9)
5. **Normalize inputs** to [-1, 1] with tanh output

### 🛡️ Common Failure Modes
| Problem | Symptom | Solution |
|---------|---------|----------|
| **Mode Collapse** | Generator produces same image repeatedly | Use minibatch discrimination, unrolled GANs |
| **Vanishing Gradients** | Generator loss doesn't decrease | Use Wasserstein loss, spectral normalization |
| **Unstable Training** | Loss oscillates wildly | Reduce learning rate, add gradient penalty |
| **Checkerboard Artifacts** | Grid patterns in generated images | Use resize-convolution instead of transposed conv |

<a id='10'></a>
## 🚀 10. Modern Variants

Since the original GAN (2014), numerous powerful variants have emerged:

### DCGAN (Deep Convolutional GAN, 2015)
- First to use deep convolutional networks successfully
- Established architectural guidelines (strided convolutions, batch norm, no fully connected layers)
- **Key Innovation**: Proved CNNs work great for GANs!

### Conditional GAN (cGAN, 2014)
- Generator and Discriminator receive additional class labels as input
- Enables **controlled generation**: "Generate a sneaker, not a bag!"
- Architecture: Concatenate label embedding to noise vector and image

### Other Notable Variants
| Variant | Year | Key Innovation |
|---------|------|---------------|
| **WGAN** | 2017 | Wasserstein loss for stable training |
| **WGAN-GP** | 2017 | Gradient penalty instead of weight clipping |
| **CycleGAN** | 2017 | Unpaired image-to-image translation |
| **StyleGAN** | 2018 | Style-based generator with progressive growing |
| **BigGAN** | 2018 | Large-scale training for high-resolution images |
| **Diffusion Models** | 2020+ | Gradually denoise random noise (state-of-the-art) |

💡 **The Evolution**: GANs → VAEs → Flow-based models → Diffusion Models. Each built on the lessons of the previous!

<a id='11'></a>
## 🛠️ Hands-On Exercises

Now it's your turn to experiment and deepen your understanding! Complete these 4 challenges:

### Exercise 1: 🎨 Latent Space Interpolation
Generate two random noise vectors and create a smooth transition between them by interpolating in the latent space. Visualize the morphing sequence.

### Exercise 2: ⚖️ Balance the Game
Modify the training loop so that the Discriminator is trained 3 times for every 1 Generator update. Observe how this affects training stability and final image quality.

### Exercise 3: 🏗️ Architecture Experiment
Change the Generator architecture: replace the final `Conv2DTranspose` layer with a combination of `UpSampling2D` + `Conv2D`. This is known to reduce checkerboard artifacts. Compare the results.

### Exercise 4: 📊 Conditional GAN Prototype
Build a simple conditional GAN by concatenating a one-hot encoded class label (0-9 for Fashion-MNIST categories) to the noise vector before feeding it into the Generator. Train it for 20 epochs and try generating specific clothing items!

<a id='12'></a>
## ✅ Solutions

Below are complete, working solutions for all four exercises. Study them carefully and compare with your implementations!

In [ ]:
# ============================================================
# ✅ SOLUTION 1: Latent Space Interpolation
# ============================================================
def interpolate_latent_space(generator, num_steps=10):
    """Create smooth transitions between two random points in latent space."""
    z1 = tf.random.normal([1, noise_dim])
    z2 = tf.random.normal([1, noise_dim])
    
    alphas = np.linspace(0, 1, num_steps)
    interpolated_images = []
    
    for alpha in alphas:
        z_interp = (1 - alpha) * z1 + alpha * z2
        img = generator(z_interp, training=False)
        interpolated_images.append(img[0])
    
    # Visualize
    plt.figure(figsize=(20, 2))
    for i, img in enumerate(interpolated_images):
        plt.subplot(1, num_steps, i + 1)
        plt.imshow(img[:, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.title(f'α={alphas[i]:.1f}', fontsize=10)
        plt.axis('off')
    plt.suptitle('🎨 Latent Space Interpolation', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

interpolate_latent_space(generator, num_steps=12)
print("✅ Solution 1 complete! Notice how the image smoothly morphs from one style to another.")

In [ ]:
# ============================================================
# ✅ SOLUTION 2: Train Discriminator 3x per Generator Update
# ============================================================
# Re-initialize models for fair comparison
generator_v2 = build_generator(noise_dim)
discriminator_v2 = build_discriminator()
gen_opt_v2 = keras.optimizers.Adam(1e-4, beta_1=0.5)
disc_opt_v2 = keras.optimizers.Adam(1e-4, beta_1=0.5)

@tf.function
def train_step_v2(images):
    """Modified training: 3 Discriminator steps per 1 Generator step."""
    # Train Discriminator 3 times
    for _ in range(3):
        noise = tf.random.normal([BATCH_SIZE, noise_dim])
        with tf.GradientTape() as disc_tape:
            generated = generator_v2(noise, training=True)
            real_out = discriminator_v2(images, training=True)
            fake_out = discriminator_v2(generated, training=True)
            d_loss = discriminator_loss(real_out, fake_out)
        disc_grads = disc_tape.gradient(d_loss, discriminator_v2.trainable_variables)
        disc_opt_v2.apply_gradients(zip(disc_grads, discriminator_v2.trainable_variables))
    
    # Train Generator 1 time
    noise = tf.random.normal([BATCH_SIZE, noise_dim])
    with tf.GradientTape() as gen_tape:
        generated = generator_v2(noise, training=True)
        fake_out = discriminator_v2(generated, training=True)
        g_loss = generator_loss(fake_out)
    gen_grads = gen_tape.gradient(g_loss, generator_v2.trainable_variables)
    gen_opt_v2.apply_gradients(zip(gen_grads, generator_v2.trainable_variables))
    
    return g_loss, d_loss

# Quick training run (10 epochs for demonstration)
print("🚀 Training with 3:1 Discriminator-to-Generator ratio...")
for epoch in range(1, 11):
    for image_batch in train_dataset:
        g_loss, d_loss = train_step_v2(image_batch)
    if epoch % 2 == 0:
        print(f"   Epoch {epoch} | Gen Loss: {g_loss:.4f} | Disc Loss: {d_loss:.4f}")

# Visualize results
test_noise = tf.random.normal([16, noise_dim])
generated_v2 = generator_v2(test_noise, training=False)
plt.figure(figsize=(8, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(generated_v2[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
    plt.axis('off')
plt.suptitle('🎨 Results: 3:1 Training Ratio', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Solution 2 complete! Compare these images with the original 1:1 ratio results.")

In [ ]:
# ============================================================
# ✅ SOLUTION 3: UpSampling2D + Conv2D Architecture
# ============================================================
def build_generator_upsample(latent_dim=100):
    """Generator using UpSampling2D + Conv2D instead of Conv2DTranspose."""
    model = keras.Sequential(name="Generator_UpSample")
    
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Reshape((7, 7, 256)))
    
    # Upsample + Conv instead of Transposed Conv
    model.add(layers.UpSampling2D(size=(2, 2)))
    model.add(layers.Conv2D(128, (3, 3), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    
    model.add(layers.UpSampling2D(size=(2, 2)))
    model.add(layers.Conv2D(64, (3, 3), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    
    model.add(layers.Conv2D(1, (3, 3), padding='same', activation='tanh', use_bias=False))
    
    return model

gen_upsample = build_generator_upsample()
gen_upsample.summary()

# Quick test
test_img = gen_upsample(tf.random.normal([1, noise_dim]), training=False)
plt.figure(figsize=(4, 4))
plt.imshow(test_img[0, :, :, 0] * 127.5 + 127.5, cmap='gray')
plt.title('🎲 UpSample Generator Output', fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()
print("✅ Solution 3 complete! This architecture typically produces smoother images with fewer artifacts.")

In [ ]:
# ============================================================
# ✅ SOLUTION 4: Simple Conditional GAN Prototype
# ============================================================
num_classes = 10
embedding_dim = 50

def build_conditional_generator(latent_dim=100, num_classes=10):
    """Conditional Generator that takes noise + class label."""
    # Noise input
    noise_input = keras.Input(shape=(latent_dim,), name='noise')
    # Label input
    label_input = keras.Input(shape=(1,), name='label')
    
    # Embed label and flatten
    label_embedding = layers.Embedding(num_classes, embedding_dim)(label_input)
    label_embedding = layers.Flatten()(label_embedding)
    
    # Concatenate noise and label embedding
    merged = layers.Concatenate()([noise_input, label_embedding])
    
    # Generator body
    x = layers.Dense(7 * 7 * 256, use_bias=False)(merged)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Reshape((7, 7, 256))(x)
    
    x = layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    x = layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    output = layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', 
                                    use_bias=False, activation='tanh')(x)
    
    model = keras.Model([noise_input, label_input], output, name="Conditional_Generator")
    return model

cgen = build_conditional_generator()
cgen.summary()

# Test conditional generation: generate specific classes
test_noise = tf.random.normal([10, noise_dim])
test_labels = tf.constant([[i] for i in range(10)])  # One of each class
conditional_images = cgen([test_noise, test_labels], training=False)

plt.figure(figsize=(15, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(conditional_images[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
    plt.title(class_names[i], fontsize=9)
    plt.axis('off')
plt.suptitle('🎯 Conditional Generation: One Sample Per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Solution 4 complete! The model architecture supports conditional generation.")
print("   💡 To fully train this, you would need to also build a Conditional Discriminator!")

<a id='13'></a>
## 🌟 Summary & Day 90 Teaser

### What You Accomplished Today 🎉
Congratulations! On Day 89, you have:
- ✅ Understood the fundamental theory behind Generative Adversarial Networks
- ✅ Built a complete Generator and Discriminator from scratch using TensorFlow/Keras
- ✅ Implemented a custom training loop with the adversarial minimax objective
- ✅ Visualized training progress and analyzed loss curves
- ✅ Learned about FID scores and GAN evaluation methodologies
- ✅ Discovered critical tips for stable GAN training
- ✅ Explored modern GAN variants including DCGAN and Conditional GANs
- ✅ Completed 4 hands-on exercises with full solutions

### Key Takeaways 💡
1. **GANs are a game**: Two networks competing drive both to improve.
2. **Stability is key**: Label smoothing, proper architectures, and balanced training are essential.
3. **Evaluation is art + science**: Combine quantitative metrics (FID) with qualitative visual inspection.
4. **The field evolves fast**: From GANs to Diffusion Models, generative AI continues to push boundaries.

### 🚀 Teaser for Day 90
Tomorrow, on **Day 90**, we will dive into **Transfer Learning with Pre-trained Models**! You'll learn how to leverage massive pre-trained networks like VGG16, ResNet, and BERT, fine-tuning them for your own tasks with a fraction of the data and compute. It's a game-changer for practical AI deployment—don't miss it!

---
⭐ **Keep pushing forward! You're 89 days into an incredible 369-day journey. The best is yet to come!** ⭐